In [1]:
#!import C:\Users\musharm\source\repos\performance_dynamic\src\benchmarks\gc\GC.Infrastructure\Notebooks\BenchmarkAnalysis_Fundamentals.ipynb

  Determining projects to restore...
  All projects are up-to-date for restore.
  GC.Analysis.API -> C:\Users\musharm\source\repos\performance_dynamic\artifacts\bin\GC.Analysis.API\Release\net8.0\GC.Analysis.API.dll

Build succeeded.
    0 Warning(s)
    0 Error(s)

Time Elapsed 00:00:01.18


Restore sources https://pkgs.dev.azure.com/dnceng/public/_packaging/dotnet-public/nuget/v3/index.json Installed Packages Microsoft.Data.Analysis, 0.19.1 Newtonsoft.Json, 13.0.3 XPlot.Plotly, 4.0.6 XPlot.Plotly.Interactive, 4.0.7 YamlDotnet, 15.1.2

Loading extensions from `Q:\.tools\.nuget\packages\microsoft.data.analysis\0.19.1\interactive-extensions\dotnet\Microsoft.Data.Analysis.Interactive.dll`

Loading extensions from `Q:\.tools\.nuget\packages\xplot.plotly.interactive\4.0.7\lib\net7.0\XPlot.Plotly.Interactive.dll`

Configuring PowerShell Kernel for XPlot.Plotly integration.

Installed support for XPlot.Plotly.

In [2]:
var fragmentationFixPath = @"C:\Users\musharm\source\repos\performance\src\benchmarks\gc\GC.Infrastructure\Configurations\ASPNetBenchmarks\FragmentationFix_Data";
var dm = DataManager.CreateAspNetData(ML(fragmentationFixPath));

In [3]:
dm

Submission#7+DataManager Data TopLevelData { Runs = System.Collections.Generic.Dictionary`2[System.String,Submission#5+RunData] } Runs key value FragmentationFix_Data RunData { Configs = System.Collections.Generic.Dictionary`2[System.String,Submission#5+ConfigData] } Configs key value datas_fix ConfigData { Benchmarks = System.Collections.Generic.Dictionary`2[System.String,Submission#5+BenchmarkData] } Benchmarks [ [ConnectionCloseHttpSys, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [FortunesEf, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [FortunesPlatformEF, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [JsonHttps, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [JsonMvc, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [MultipleQueriesPlatform, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [PlaintextMvc, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [PlaintextWithParametersNoFilter, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [SingleQueryPlatform, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [Stage1Pgo, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [UpdatesPlatform, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }] ] datas_nofix ConfigData { Benchmarks = System.Collections.Generic.Dictionary`2[System.String,Submission#5+BenchmarkData] } Benchmarks [ [ConnectionCloseHttpSys, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [FortunesEf, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [FortunesPlatformEF, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [JsonHttps, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [JsonMvc, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [MultipleQueriesPlatform, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [PlaintextMvc, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [PlaintextWithParametersNoFilter, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [SingleQueryPlatform, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [Stage1Pgo, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [UpdatesPlatform, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }] ] server_fix ConfigData { Benchmarks = System.Collections.Generic.Dictionary`2[System.String,Submission#5+BenchmarkData] } Benchmarks [ [ConnectionCloseHttpSys, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [FortunesEf, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic.List`1[Submission#5+IterationData] }], [FortunesPlatformEF, BenchmarkData { SummaryLoadInfo = , Iterations = System.Collections.Generic

In [4]:
static Metric<TraceGC> excessFragmentation = new Metric<TraceGC>((d => {
    Dictionary<CondemnedReasonGroup, int> condemnReasonsPerGC = new Dictionary<CondemnedReasonGroup, int>();
    d.GetCondemnedReasons(condemnReasonsPerGC);
    return condemnReasonsPerGC.ContainsKey(CondemnedReasonGroup.Fragmented_Ephemeral) ? 1 : 0; 
}), "IsExcessEphFrag", "#");
static Aggregation sumAggregation = new Aggregation((d => d.Sum()), "Sum", "#");
static Metric<IterationData> iterationExcessFragmentation = Metrics.Promote(excessFragmentation, sumAggregation);
static Metric<BenchmarkData> TotalExcessFragmentation = Metrics.Promote(iterationExcessFragmentation, Aggregation.Average);

### Existence of Issue in Fix

In [5]:
// Num of GCs done.
// Individual GCs.
// Volatility within the Runs.
TableBenchmarks(dm, ML( TotalExcessFragmentation ), configFilter: new Filter( ML ( "datas_fix", "datas_nofix", "server_fix" )), compareInfo: new CompareInfo("datas_nofix", "datas_fix"), textPresenter: new MarkdownPresenter());

### Per-benchmark behavior / Average of Sum of IsExcessEphFrag

|                  Benchmark Name | datas_nofix | datas_fix |  Comparison % |
| ------------------------------: | ----------: | --------: | ------------: |
|          ConnectionCloseHttpSys |       3.250 |     0.000 |      -100.000 |
|                      FortunesEf |       2.500 |     0.000 |      -100.000 |
|              FortunesPlatformEF |       1.750 |     0.000 |      -100.000 |
|                       JsonHttps |       0.250 |     0.000 |      -100.000 |
|                         JsonMvc |       0.500 |     0.000 |      -100.000 |
|         MultipleQueriesPlatform |       2.250 |     0.000 |      -100.000 |
|                    PlaintextMvc |   1,104.000 |     0.000 |      -100.000 |
| PlaintextWithParametersNoFilter |     437.250 |     0.000 |      -100.000 |
|             SingleQueryPlatform |       1.000 |     0.000 |      -100.000 |
|                       Stage1Pgo |      97.500 |     0.000 |      -100.000 |
|                 UpdatesPlatform |      76.750 |     0.000 |      -100.000 |

### Server Fix vs. DATAS Fix

In [15]:
var t = TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency),
 compareInfo: new CompareInfo("datas_nofix", "datas_fix"), textPresenter: new MarkdownPresenter() );
string.Join("\n", t[0]).Display();

### Per-benchmark behavior

|                  Benchmark Name | Average of Max heap size / datas_nofix | Average of Max heap size / datas_fix | Average of Max heap size /  Comparison % | Average of RPS / datas_nofix | Average of RPS / datas_fix | Average of RPS /  Comparison % | Average of Latency 50th / datas_nofix | Average of Latency 50th / datas_fix | Average of Latency 50th /  Comparison % |
| ------------------------------: | -------------------------------------: | -----------------------------------: | ---------------------------------------: | ---------------------------: | -------------------------: | -----------------------------: | ------------------------------------: | ----------------------------------: | --------------------------------------: |
|          ConnectionCloseHttpSys |                                 17.526 |                               20.692 |                                   18.064 |                      104.108 |                    104.480 |                          0.357 |                                 0.145 |                               0.143 |                                  -1.724 |
|                      FortunesEf |                                 83.147 |                               80.816 |                                   -2.804 |                      282.034 |                    282.235 |                          0.071 |                                 0.762 |                               0.763 |                                   0.000 |
|              FortunesPlatformEF |                                141.925 |                              140.192 |                                   -1.221 |                      388.127 |                    390.874 |                          0.708 |                                 1.180 |                               1.177 |                                  -0.212 |
|                       JsonHttps |                                 48.916 |                               56.748 |                                   16.012 |                      703.805 |                    689.928 |                         -1.972 |                                 0.320 |                               0.328 |                                   2.344 |
|                         JsonMvc |                                 42.326 |                               42.501 |                                    0.412 |                      610.478 |                    609.436 |                         -0.171 |                                 0.370 |                               0.373 |                                   0.676 |
|         MultipleQueriesPlatform |                                115.685 |                              113.330 |                                   -2.036 |                       44.379 |                     44.126 |                         -0.571 |                                10.870 |                              10.920 |                                   0.460 |
|                    PlaintextMvc |                                 40.356 |                               34.742 |                                  -13.911 |                    2,942.056 |                  2,879.843 |                         -2.115 |                                 0.775 |                               0.788 |                                   1.613 |
| PlaintextWithParametersNoFilter |                                 44.692 |                               47.045 |                                    5.266 |                    3,595.981 |                  3,566.217 |                         -0.828 |                                 0.647 |                               0.652 |                                   0.772 |
|             SingleQueryPlatform |                                 85.443 |                               83.003 |                                   -2.856 |                      538.835 |                    537.947 |                         -0.165 |                                 0.787 |                               0.785 |                                  -0.317 |
|                       Stage1Pgo |                                 35.519 |                               34.368 |                                   -3.240 |                      766.418 |                    768.125 |                          0.223 |                                 0.295 |                               0.297 |                                   0.847 |
|                 UpdatesPlatform |                                108.229 |                              107.551 |                                   -0.627 |                       27.084 |                     28.040 |                          3.530 |                                17.640 |                              17.352 |                                  -1.630 |

### Per-benchmark behavior

|                  Benchmark Name | Average of Max heap size / datas_nofix | Average of Max heap size / datas_fix | Average of Max heap size /  Comparison % | Average of RPS / datas_nofix | Average of RPS / datas_fix | Average of RPS /  Comparison % | Average of Latency 50th / datas_nofix | Average of Latency 50th / datas_fix | Average of Latency 50th /  Comparison % |
| ------------------------------: | -------------------------------------: | -----------------------------------: | ---------------------------------------: | ---------------------------: | -------------------------: | -----------------------------: | ------------------------------------: | ----------------------------------: | --------------------------------------: |
|          ConnectionCloseHttpSys |                                 17.526 |                               20.692 |                                   18.064 |                      104.108 |                    104.480 |          

In [8]:
var t = TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency),
 compareInfo: new CompareInfo("server_fix", "datas_fix"), textPresenter: new MarkdownPresenter() );

// Rename CompareInfo -> ConfigCompareInfo

### Per-benchmark behavior

|                  Benchmark Name | Average of Max heap size / server_fix | Average of Max heap size / datas_fix | Average of Max heap size /  Comparison % | Average of RPS / server_fix | Average of RPS / datas_fix | Average of RPS /  Comparison % | Average of Latency 50th / server_fix | Average of Latency 50th / datas_fix | Average of Latency 50th /  Comparison % |
| ------------------------------: | ------------------------------------: | -----------------------------------: | ---------------------------------------: | --------------------------: | -------------------------: | -----------------------------: | -----------------------------------: | ----------------------------------: | --------------------------------------: |
|          ConnectionCloseHttpSys |                               352.582 |                               20.692 |                                  -94.131 |                     107.925 |                    104.480 |                         -3.191 |                                0.140 |                               0.143 |                                   1.786 |
|                      FortunesEf |                               400.998 |                               80.816 |                                  -79.846 |                     296.430 |                    282.235 |                         -4.789 |                                0.740 |                               0.763 |                                   3.041 |
|              FortunesPlatformEF |                               491.095 |                              140.192 |                                  -71.453 |                     404.066 |                    390.874 |                         -3.265 |                                1.165 |                               1.177 |                                   1.073 |
|                       JsonHttps |                               340.241 |                               56.748 |                                  -83.321 |                     715.269 |                    689.928 |                         -3.543 |                                0.325 |                               0.328 |                                   0.769 |
|                         JsonMvc |                               349.499 |                               42.501 |                                  -87.840 |                     638.686 |                    609.436 |                         -4.580 |                                0.377 |                               0.373 |                                  -1.325 |
|         MultipleQueriesPlatform |                               395.043 |                              113.330 |                                  -71.312 |                      44.358 |                     44.126 |                         -0.524 |                               10.870 |                              10.920 |                                   0.460 |
|                    PlaintextMvc |                               352.589 |                               34.742 |                                  -90.147 |                   3,278.344 |                  2,879.843 |                        -12.156 |                                0.718 |                               0.788 |                                   9.756 |
| PlaintextWithParametersNoFilter |                               347.435 |                               47.045 |                                  -86.459 |                   3,773.247 |                  3,566.217 |                         -5.487 |                                0.627 |                               0.652 |                                   3.984 |
|             SingleQueryPlatform |                               372.059 |                               83.003 |                                  -77.691 |                     573.832 |                    537.947 |                         -6.254 |                                0.702 |                               0.785 |                                  11.744 |
|                       Stage1Pgo |                               345.363 |                               34.368 |                                  -90.049 |                     792.528 |                    768.125 |                         -3.079 |                                0.292 |                               0.297 |                                   1.709 |
|                 UpdatesPlatform |                               388.802 |                              107.551 |                                  -72.338 |                      27.909 |                     28.040 |                          0.471 |                               17.393 |                              17.352 |                                  -0.230 |

### Server No Fix vs. DATAS No Fix

In [9]:
TableBenchmarks(dm, ML( Metrics.B.AverageMaxHeapSize, Metrics.B.AverageRequestPerMSec, Metrics.B.AverageP50Latency), configFilter: new Filter( ML ( "server_nofix", "datas_nofix" )),
 compareInfo: new CompareInfo("server_nofix", "datas_nofix"), textPresenter: new MarkdownPresenter() );

### Per-benchmark behavior

|                  Benchmark Name | Average of Max heap size / server_nofix | Average of Max heap size / datas_nofix | Average of Max heap size /  Comparison % | Average of RPS / server_nofix | Average of RPS / datas_nofix | Average of RPS /  Comparison % | Average of Latency 50th / server_nofix | Average of Latency 50th / datas_nofix | Average of Latency 50th /  Comparison % |
| ------------------------------: | --------------------------------------: | -------------------------------------: | ---------------------------------------: | ----------------------------: | ---------------------------: | -----------------------------: | -------------------------------------: | ------------------------------------: | --------------------------------------: |
|          ConnectionCloseHttpSys |                                 352.643 |                                 17.526 |                                  -95.030 |                       107.991 |                      104.108 |                         -3.595 |                                  0.140 |                                 0.145 |                                   3.571 |
|                      FortunesEf |                                 401.153 |                                 83.147 |                                  -79.273 |                       302.810 |                      282.034 |                         -6.861 |                                  0.730 |                                 0.762 |                                   4.452 |
|              FortunesPlatformEF |                                 659.066 |                                141.925 |                                  -78.466 |                       401.039 |                      388.127 |                         -3.220 |                                  1.172 |                                 1.180 |                                   0.640 |
|                       JsonHttps |                                 340.252 |                                 48.916 |                                  -85.624 |                       723.839 |                      703.805 |                         -2.768 |                                  0.320 |                                 0.320 |                                   0.000 |
|                         JsonMvc |                                 349.079 |                                 42.326 |                                  -87.875 |                       642.278 |                      610.478 |                         -4.951 |                                  0.370 |                                 0.370 |                                   0.000 |
|         MultipleQueriesPlatform |                                 400.301 |                                115.685 |                                  -71.100 |                           NaN |                       44.379 |                            NaN |                                    NaN |                                10.870 |                                     NaN |
|                    PlaintextMvc |                                 355.915 |                                 40.356 |                                  -88.661 |                     3,304.144 |                    2,942.056 |                        -10.959 |                                  0.712 |                                 0.775 |                                   8.772 |
| PlaintextWithParametersNoFilter |                                 347.444 |                                 44.692 |                                  -87.137 |                     3,733.891 |                    3,595.981 |                         -3.693 |                                  0.625 |                                 0.647 |                                   3.600 |
|             SingleQueryPlatform |                                 372.607 |                                 85.443 |                                  -77.069 |                       574.731 |                      538.835 |                         -6.246 |                                  0.707 |                                 0.787 |                                  11.307 |
|                       Stage1Pgo |                                 343.956 |                                 35.519 |                                  -89.673 |                       789.263 |                      766.418 |                         -2.895 |                                  0.295 |                                 0.295 |                                   0.000 |
|                 UpdatesPlatform |                                 394.328 |                                108.229 |                                  -72.553 |                        28.362 |                       27.084 |                         -4.504 |                                 17.205 |                                17.640 |                                   2.528 |